In [1]:
# This file document the code of calculating skill similarity matrix between occupations and then convert the matrix form to long panel data in "/Data/use/panel_jskill_sim.csv" for later use.

import pandas as pd
import numpy as np
from itertools import combinations

print("=" * 80)
print("Skill Similarity Calculation - Complete Workflow (Within-Occupation Normalization)")
print("=" * 80)

# ==================== Part 1: Calculate Similarity Matrix ====================

print("\n【Part 1: Calculate Similarity Matrix】\n")

print("Step 1: Load data")
file_path = r"Your path/Stranded_Occupations_Replication/Data/raw/skill_v3.xlsx"
data = pd.read_excel(file_path)
data.columns = ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Value']

print(f"✓ Raw data shape: {data.shape}")
print(f"✓ Number of occupations: {data['Title'].nunique()}")
print(f"✓ Number of skills: {data['Element Name'].nunique()}")

print("\n Step 2: Within-occupation normalization (normalize each occupation's skill importance to [0, 1])")
# Note: Using within-occupation normalization
data['Normalized Value'] = data.groupby('Title')['Value'].transform(
    lambda x: (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else 0
)

print(f"✓ Normalized range: [{data['Normalized Value'].min():.2f}, {data['Normalized Value'].max():.2f}]")

print("\nStep 3: Build pivot table")
# Build pivot table: rows are occupations, columns are skills, values are normalized importance
pivot = data.pivot(index='Title', columns='Element Name', values='Normalized Value').fillna(0)

print(f"✓ Pivot table shape: {pivot.shape}")
print(f"✓ Number of occupations: {len(pivot.index)}")
print(f"✓ Number of skills: {len(pivot.columns)}")

print("\nStep 4: Calculate Jaccard similarity (continuous version)")
occupations = pivot.index
pairs = combinations(occupations, 2)

# Calculate using Jaccard similarity formula
similarities = []
pair_count = 0
total_pairs = len(occupations) * (len(occupations) - 1) // 2

print(f"✓ Total pairs to calculate: {total_pairs}")
print("Calculation progress:")

for occ1, occ2 in pairs:
    skills_1 = pivot.loc[occ1].values
    skills_2 = pivot.loc[occ2].values
    similarity = np.sum(np.minimum(skills_1, skills_2)) / np.sum(np.maximum(skills_1, skills_2))
    similarities.append((occ1, occ2, similarity))
    
    pair_count += 1
    if pair_count % 20000 == 0:
        print(f"  {pair_count}/{total_pairs} ({100*pair_count/total_pairs:.1f}%)")

print(f"✓ Done! Calculated {len(similarities)} occupation pairs")

print("\nStep 5: Convert to matrix form")
# Store results in DataFrame
similarity_df = pd.DataFrame(similarities, columns=['Occupation 1', 'Occupation 2', 'skill Similarity'])

# Convert similarity results to matrix form
similarity_matrix = pd.pivot_table(
    similarity_df, 
    index='Occupation 1', 
    columns='Occupation 2', 
    values='skill Similarity'
)

print(f"✓ Initial matrix shape: {similarity_matrix.shape}")

# Matrix needs to be symmetric, fill upper triangle
similarity_matrix = similarity_matrix.fillna(similarity_matrix.T)

print(f"✓ Shape after symmetric filling: {similarity_matrix.shape}")

print("\nStep 6: Save matrix file")
# Save matrix results to CSV file
output_path = r"Your path/Stranded_Occupations_Replication/Data/temp/jskill_sim.csv"
similarity_matrix.to_csv(output_path)
print(f"✓ Saved to: {output_path}")

# ==================== Part 2: Convert to Panel Data ====================

print("\n" + "=" * 80)
print("【Part 2: Convert to Panel Data】\n")

print("Step 7: Load matrix data")
# Assume matrix data is saved in "matrix_data.csv"
matrix_data = pd.read_csv(output_path, index_col=0)

# Check dataframe structure
print("DataFrame Head:")
print(matrix_data.head())
print("Columns:", matrix_data.columns.tolist()[:5], "...")

print("\nStep 8: Reset index")
# Reset index to regular column
matrix_data = matrix_data.reset_index()

# Confirm column names after reset
print("After reset_index Columns:", matrix_data.columns.tolist()[:5], "...")

# Check first column name
first_col = matrix_data.columns[0]
if first_col != 'Occupation 1':
    matrix_data.rename(columns={first_col: 'Occupation 1'}, inplace=True)
    print(f"✓ Renamed '{first_col}' to 'Occupation 1'")

print("\nStep 9: Convert to long format (panel data)")
# Convert matrix to panel data
panel_data = matrix_data.melt(
    id_vars=["Occupation 1"],  # Modify to actual column name
    var_name="match_occupation",  # Column name becomes match_occupation
    value_name="skill_similarity"  # Cell values are skill_similarity
)

# Rename columns
panel_data.rename(columns={"Occupation 1": "occupation"}, inplace=True)

print(f"✓ Panel data shape: {panel_data.shape}")
print(f"✓ Column names: {panel_data.columns.tolist()}")

print("\nStep 10: Check specific missing data")
# Note: The converted panel data is missing the pair with Zoologists and Wildlife Biologists as occupation
# and Accountants and Auditors as match_occupation

# Check if this pair exists
check_pair = panel_data[
    (panel_data['occupation'] == 'Zoologists and Wildlife Biologists') & 
    (panel_data['match_occupation'] == 'Accountants and Auditors')
]

if len(check_pair) == 0:
    print("⚠️ Missing found: Zoologists and Wildlife Biologists -> Accountants and Auditors")
    
    # Find data at symmetric position
    symmetric_pair = panel_data[
        (panel_data['occupation'] == 'Accountants and Auditors') & 
        (panel_data['match_occupation'] == 'Zoologists and Wildlife Biologists')
    ]
    
    if len(symmetric_pair) > 0:
        similarity_value = symmetric_pair['skill_similarity'].values[0]
        print(f"  Found similarity from symmetric position: {similarity_value:.6f}")
        
        # Add this data
        new_row = pd.DataFrame({
            'occupation': ['Zoologists and Wildlife Biologists'],
            'match_occupation': ['Accountants and Auditors'],
            'skill_similarity': [similarity_value]
        })
        panel_data = pd.concat([panel_data, new_row], ignore_index=True)
        print("  ✓ Data supplemented")
    else:
        print("  ❌ No data at symmetric position either!")
else:
    print("✓ This data pair already exists")

print("\nStep 11: Comprehensive check for missing data")
# Check if there are other missing symmetric data
all_occupations = set(occupations)
existing_pairs = set(zip(panel_data['occupation'], panel_data['match_occupation']))

missing_pairs = []
for occ1 in all_occupations:
    for occ2 in all_occupations:
        if (occ1, occ2) not in existing_pairs:
            missing_pairs.append((occ1, occ2))

if missing_pairs:
    print(f"⚠️ Found total {len(missing_pairs)} missing data pairs")
    print(f"  Examples (first 5):")
    for pair in missing_pairs[:5]:
        print(f"    {pair[0]} -> {pair[1]}")
    
    # Supplement all missing data
    print("\nStep 12: Supplement all missing data")
    missing_data = []
    for occ1, occ2 in missing_pairs:
        # Get similarity from symmetric position
        symmetric_match = panel_data[
            (panel_data['occupation'] == occ2) & 
            (panel_data['match_occupation'] == occ1)
        ]
        
        if len(symmetric_match) > 0:
            similarity = symmetric_match['skill_similarity'].values[0]
        else:
            # If same occupation, similarity is 1
            similarity = 1.0 if occ1 == occ2 else 0.0
        
        missing_data.append({
            'occupation': occ1,
            'match_occupation': occ2,
            'skill_similarity': similarity
        })
    
    missing_df = pd.DataFrame(missing_data)
    panel_data = pd.concat([panel_data, missing_df], ignore_index=True)
    print(f"✓ Supplemented {len(missing_data)} data entries")
else:
    print("✓ No missing data")

print(f"\n✓ Final panel data shape: {panel_data.shape}")

print("\nStep 13: Save panel data")
# Save converted data
output_path = r"Your path/Stranded_Occupations_Replication/Data/use/panel_jskill_sim.csv"
panel_data.to_csv(output_path, index=False)

print(f"✓ Conversion successful, file saved to: {output_path}")

# ==================== Part 3: Validation ====================

print("\n" + "=" * 80)
print("【Part 3: Validation】\n")

print("Validation 1: Check panel data completeness")
n_occupations = len(all_occupations)
expected_rows = n_occupations * n_occupations
actual_rows = len(panel_data)
print(f"✓ Number of occupations: {n_occupations}")
print(f"✓ Expected rows: {expected_rows} ({n_occupations} × {n_occupations})")
print(f"✓ Actual rows: {actual_rows}")
print(f"✓ Complete: {'✅ Yes' if expected_rows == actual_rows else '❌ No'}")

print("\nValidation 2: Recheck specific data pair")
check_pair = panel_data[
    (panel_data['occupation'] == 'Zoologists and Wildlife Biologists') & 
    (panel_data['match_occupation'] == 'Accountants and Auditors')
]
print(f"✓ Zoologists -> Accountants data: {len(check_pair)} entries")
if len(check_pair) > 0:
    print(f"  Similarity: {check_pair['skill_similarity'].values[0]:.6f}")

print("\nValidation 3: Statistical information")
print(f"✓ Similarity range: [{panel_data['skill_similarity'].min():.6f}, {panel_data['skill_similarity'].max():.6f}]")
print(f"✓ Mean similarity: {panel_data['skill_similarity'].mean():.6f}")
print(f"✓ Median similarity: {panel_data['skill_similarity'].median():.6f}")

print("\nValidation 4: Preview panel data")
print(panel_data.head(10))

print("\n" + "=" * 80)
print("✅ All steps completed!")
print("=" * 80)


Skill Similarity Calculation - Complete Workflow (Within-Occupation Normalization)

【Part 1: Calculate Similarity Matrix】

Step 1: Load data
✓ Raw data shape: (31290, 5)
✓ Number of occupations: 894
✓ Number of skills: 35

 Step 2: Within-occupation normalization (normalize each occupation's skill importance to [0, 1])
✓ Normalized range: [0.00, 1.00]

Step 3: Build pivot table
✓ Pivot table shape: (894, 35)
✓ Number of occupations: 894
✓ Number of skills: 35

Step 4: Calculate Jaccard similarity (continuous version)
✓ Total pairs to calculate: 399171
Calculation progress:
  20000/399171 (5.0%)
  40000/399171 (10.0%)
  60000/399171 (15.0%)
  80000/399171 (20.0%)
  100000/399171 (25.1%)
  120000/399171 (30.1%)
  140000/399171 (35.1%)
  160000/399171 (40.1%)
  180000/399171 (45.1%)
  200000/399171 (50.1%)
  220000/399171 (55.1%)
  240000/399171 (60.1%)
  260000/399171 (65.1%)
  280000/399171 (70.1%)
  300000/399171 (75.2%)
  320000/399171 (80.2%)
  340000/399171 (85.2%)
  360000/399171 (